In [18]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [19]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

def get_client():
    return OpenAI(api_key=api_key)

client = get_client()

API key found and looks good so far!


In [20]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are an assistant that analyzes the contents of a website,
and provides a short summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [21]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

In [22]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [23]:
model = "gpt-4.1-mini" #defining them upfront to make it easier to change later if you want to experiment with different models
max_tokens = 500
llm_provider = "openai"
url = "https://openai.com"

def llm_chat(messages):
    if llm_provider == "openai":
        response = client.chat.completions.create(
            model=model,
            max_tokens=max_tokens,
            messages=messages
        )
        return response.choices[0].message.content
    else:
        raise ValueError(f"Unknown LLM provider: {llm_provider}")

def split_text_into_chunks(text, chunk_size=3000):
    return [
        text[i:i + chunk_size] 
        for i in range(0, len(text), chunk_size)
    ]

def summarize(url):
    content = fetch_website_contents(url)

    # If content is small -> simple path
    if len(content) < 4000:
        return llm_chat(messages_for(content))
    
    #If content is large -> split into chunks and summarize each chunk, then summarize the summaries
    chunks = split_text_into_chunks(content)
    chunk_summaries = []
    for chunk in chunks:
        summary = llm_chat(messages_for(chunk))
        chunk_summaries.append(summary)

    combined_summary = "\n\n".join(chunk_summaries)
    return llm_chat(messages_for(combined_summary))


def display_summary(summary):
    display(Markdown(summary))

summary = summarize(url)
display_summary(summary)

# OpenAI Website Summary

The OpenAI website focuses on their latest advancements in artificial intelligence research, product development, and business applications. Key highlights include:

- **Research & Products:** Detailed information on cutting-edge AI models like GPT-5.6 and GPT-Live, emphasizing their potential to scale with user ambition and support complex tasks.
- **Recent News & Announcements:**
  - Submitted a confidential draft S-1 to the SEC (June 8, 2026).
  - Released measures to enhance content safety and provenance in AI ecosystems.
  - Launched new features such as a personal finance experience in ChatGPT.
  - Emphasis on agent technologies transforming work environments.
- **Features & Use Cases:** Examples of AI in practice, including collaborations like Chip Ganassi Racing, simulations of black holes, and AI-driven food distribution automation.
- **Research Contributions:** Publication of significant research such as disproving a key conjecture in discrete geometry and introducing GPT-Rosalind for life sciences.
- **Safety and Monitoring:** Focus on monitoring AI agents for misalignment and handling sensitive conversations more effectively.
- **Business & Enterprise Solutions:** Offering ChatGPT Enterprise and Codex to enhance productivity in industries like banking and food distribution.

The site also provides resources for developers, startup integrations, and an overview of economic research related to AI.